# 11 · Trained baselines 1/2: training data and all test texts (design doc §8.8.16)

**How to run**: run the first cell to free GPU memory; pick a Colab server with an **A100 GPU** and click Run All; about 30 minutes. A rerun after a disconnect skips finished parts. Results go to `DATA_DIR/11_trained_baselines/`.

**What it computes**: 1,000 more unused MAGE human texts are drawn, and three generators continue the first 300 at four temperatures (3,600 texts) as training data for the trained detectors. The training texts and all test texts (main1, external validation A, M4GT B) are then collected in one table, `texts.csv`, for the second notebook.

The `set` column of `texts.csv` keeps the study labels: `main1` = `06_scaled_sweep`, `extA` = `09_heldout_pythia`, `m4gt` = `08_m4gt_natural_text`, `train` = the training texts drawn in this notebook.

In [ ]:
# 1. Check the GPU. Two 7B models need about 30 GB free.
import torch
assert torch.cuda.is_available(), "No GPU: select an A100 runtime."
free = torch.cuda.mem_get_info()[0] / 2**30
print(torch.cuda.get_device_name(0), f"{free:.0f} GB free")
assert free > 30, "Not enough free GPU memory; restart the runtime."


In [ ]:
# 2. Google Drive and packages
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.makedirs("11_trained_baselines", exist_ok=True); os.environ["HF_HOME"] = "/content/hf"
%pip install -q -U transformers huggingface-hub accelerate
import json, gc, glob, shutil, hashlib, csv
for _d in ("/root/.cache/huggingface", "/content/hf"):   # clear models downloaded by other notebooks on this machine (the two Falcons alone take 28 GB); needed ones are downloaded again
    shutil.rmtree(_d, ignore_errors=True)
free_gb = shutil.disk_usage("/content").free / 2**30; print(f"{free_gb:.0f} GB free on the machine disk")
assert free_gb > 20, "Not enough disk space on the machine: Runtime → Disconnect and delete runtime, then Run All"
import numpy as np, pandas as pd


In [ ]:
# 3. Draw 1,000 more MAGE human texts, none of the 1,000 in 06_scaled_sweep or the 100 pilot texts; 120 to 300 words; seed 2
DOMAINS = ["xsum", "eli5", "yelp", "wp", "sci_gen", "cmv", "squad"]
if not os.path.exists("11_trained_baselines/humans_train_1000.json"):
    if not os.path.exists("/content/test.csv"):
        !curl -sSL --retry 5 -o /content/test.csv https://huggingface.co/datasets/yaful/MAGE/resolve/main/test.csv
    d = pd.read_csv("/content/test.csv").dropna(subset=["text"]).drop_duplicates("text")
    used = {h["text"] for h in json.load(open("06_scaled_sweep/humans_1000.json"))} | {h["text"] for h in json.load(open("01_pilot_decoding/humans.json"))}
    d = d[d.src.isin([f"{x}_human" for x in DOMAINS]) & ~d.text.isin(used)]
    words = d.text.str.split().str.len(); d = d[(words >= 120) & (words <= 300)]
    per = {dom: 143 for dom in DOMAINS}; per["xsum"] = 142
    pick = pd.concat([d[d.src == f"{dom}_human"].sample(n, random_state=2) for dom, n in per.items()]).sample(frac=1, random_state=2)
    rows = [dict(id=i, domain=s.replace("_human", ""), text=t) for i, (s, t) in enumerate(zip(pick.src, pick.text))]
    json.dump(rows, open("11_trained_baselines/humans_train_1000.json", "w"), ensure_ascii=False)
humans = json.load(open("11_trained_baselines/humans_train_1000.json")); sources = humans[:300]
assert len(humans) == 1000 and not ({h["text"] for h in humans} & {h["text"] for h in json.load(open("06_scaled_sweep/humans_1000.json"))})
print(len(humans), "training human texts; prompts:", len(sources), pd.Series([h["domain"] for h in sources]).value_counts().to_dict())


In [ ]:
# 4. Three generators x four temperatures continue 300 texts (rules as in 06; seed 43)
from transformers import AutoTokenizer, AutoModelForCausalLM
GENS = {"Qwen/Qwen2.5-3B": [0.8, 0.9, 1.0, 1.1], "microsoft/phi-2": [0.8, 0.9, 1.0, 1.1], "HuggingFaceTB/SmolLM2-1.7B": [0.8, 0.9, 1.0, 1.1]}
PREFIX, MAX_TOK, BS, SEED = 30, 512, 24, 43

def prepare(tok, texts):
    items = []
    for h in texts:
        ids = tok(h["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) < PREFIX + 20:
            continue
        items.append(dict(id=h["id"], domain=h["domain"], prefix_ids=ids[:PREFIX], target=len(ids) - PREFIX,
                          human=tok.decode(ids[PREFIX:], clean_up_tokenization_spaces=False), prefix=tok.decode(ids[:PREFIX], clean_up_tokenization_spaces=False)))
    return items

@torch.no_grad()
def generate_batch(model, tok, batch, temp, special):
    pad = tok.pad_token_id
    L = max(len(b["prefix_ids"]) for b in batch); new = max(b["target"] for b in batch)
    x = torch.full((len(batch), L), pad); att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        x[i, L - len(b["prefix_ids"]):] = torch.tensor(b["prefix_ids"]); att[i, L - len(b["prefix_ids"]):] = 1
    out = model.generate(input_ids=x.cuda(), attention_mask=att.cuda(), do_sample=True, temperature=temp, top_p=1.0, top_k=0,
                         max_new_tokens=new, min_new_tokens=new, suppress_tokens=special, pad_token_id=pad, repetition_penalty=1.0)
    gen = out[:, L:].cpu(); res = [gen[i, :b["target"]].tolist() for i, b in enumerate(batch)]
    for b, ids in zip(batch, res):
        assert len(ids) == b["target"] and not (set(ids) & set(special))
    return res

for gen_name, temps in GENS.items():
    tag = gen_name.split("/")[1]
    todo = [t for t in temps if not os.path.exists(f"11_trained_baselines/gen_{tag}_T{t}.json")]; hpath = f"11_trained_baselines/humans_train_{tag}.json"
    if not todo and os.path.exists(hpath):
        print("Already done, skipping:", tag); continue
    globals().pop("model", None); gc.collect(); torch.cuda.empty_cache()
    tok = AutoTokenizer.from_pretrained(gen_name); tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(gen_name, dtype=torch.bfloat16).cuda().eval()
    special = sorted(set(tok.all_special_ids) | {tok.eos_token_id} | set(np.atleast_1d(model.generation_config.eos_token_id).tolist() if model.generation_config.eos_token_id is not None else []))
    items = sorted(prepare(tok, sources), key=lambda b: b["target"]); assert len(items) == 300
    for temp in todo:
        torch.manual_seed(SEED); t0 = time.time(); rows = []
        for s in range(0, len(items), BS):
            batch = items[s:s + BS]
            for b, ids in zip(batch, generate_batch(model, tok, batch, temp, special)):
                rows.append(dict(id=b["id"], domain=b["domain"], model=gen_name, decoding=f"T{temp}", prefix=b["prefix"], ai=tok.decode(ids, clean_up_tokenization_spaces=False), human=b["human"], n_tokens=len(ids)))
        json.dump(rows, open(f"11_trained_baselines/gen_{tag}_T{temp}.json", "w"), ensure_ascii=False)
        print(f"{tag} T={temp}: {len(rows)} texts, {(time.time() - t0) / 60:.1f} min", flush=True)
    if not os.path.exists(hpath):
        hum = prepare(tok, humans); assert len(hum) == 1000
        json.dump([dict(id=b["id"], domain=b["domain"], prefix=b["prefix"], human=b["human"]) for b in hum], open(hpath, "w"), ensure_ascii=False)
    del model; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(f"/content/hf/hub/models--{gen_name.replace('/', '--')}", ignore_errors=True)   # delete after use; models live under hub/, so the path must include it


In [ ]:
# 5. Build one table, texts.csv: training texts + three test sets, continuation parts only (whole texts for M4GT B)
rows = []
def add(setname, recs, who, model, dec_key=None, text_key="ai"):
    for r in recs:
        rows.append(dict(set=setname, id=r["id"], domain=r["domain"], model=model, decoding=(r[dec_key] if dec_key else "human"), who=who, text=r[text_key]))
# training: humans with the Qwen split; AI texts from three models at four temperatures
add("train", json.load(open("11_trained_baselines/humans_train_Qwen2.5-3B.json")), "human", "human", text_key="human")
for f in sorted(glob.glob("11_trained_baselines/gen_*_T*.json")):
    add("train", json.load(open(f)), "ai", os.path.basename(f).split("_T")[0].replace("gen_", ""), "decoding")
# main1
add("main1", json.load(open("06_scaled_sweep/humans_Qwen2.5-3B.json")), "human", "human", text_key="human")
for f in sorted(glob.glob("06_scaled_sweep/gen_*_T*.json")):
    add("main1", json.load(open(f)), "ai", os.path.basename(f).split("_T")[0].replace("gen_", ""), "decoding")
# external validation A
add("extA", json.load(open("09_heldout_pythia/humans_pythia-2.8b.json")), "human", "human", text_key="human")
for f in sorted(glob.glob("09_heldout_pythia/gen_pythia-2.8b_T*.json")):
    add("extA", json.load(open(f)), "ai", "pythia-2.8b", "decoding")
# M4GT B: read the raw texts listed in the frozen manifest
man = pd.read_csv("manifests/m4gt_subtaskb_manifest.csv"); need = {int(r.line_number): r for r in man.itertuples(index=False)}
with open("external/m4gt/SubtaskB.jsonl", encoding="utf-8") as f:
    for ln, line in enumerate(f, start=1):
        r = need.get(ln)
        if r is None: continue
        it = json.loads(line); assert hashlib.sha256(it["text"].encode("utf-8")).hexdigest() == r.text_sha256
        rows.append(dict(set="m4gt", id=ln, domain=r.source, model=r.model, decoding=("human" if int(r.label) == 1 else "natural"), who=("human" if int(r.label) == 1 else "ai"), text=it["text"]))
T = pd.DataFrame(rows)
print(T.groupby(["set", "who"]).size().to_string())
assert (T.groupby("set").size().to_dict() == {"train": 4600, "main1": 9100, "extA": 3700, "m4gt": 8400}), T.groupby("set").size().to_dict()
T.to_csv("11_trained_baselines/texts.csv", index=False)
info = dict(rows=len(T), sha256=hashlib.sha256(open("11_trained_baselines/texts.csv", "rb").read()).hexdigest(), human_seed=2, gen_seed=43, per_set=T.groupby("set").size().to_dict())
json.dump(info, open("11_trained_baselines/data_info.json", "w"), ensure_ascii=False, indent=2); print(info)


In [ ]:
%cd /content
drive.flush_and_unmount(); print("Done: results are in DATA_DIR/11_trained_baselines/; next, run 12_trained_baselines_run.ipynb")
